# 08 Advanced Challenge - Data QC and Outlier Visualization in Python

## Biochemistry question

In this synthetic assay dataset, which measurements should be reviewed before interpreting the overall pattern?


In [1]:
import plotly.io as pio
pio.renderers.default = "iframe"


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/qc_outlier_sample.csv")
df.head()

,sample_id,drug_name,concentration_uM,replicate,cell_viability_percent
0,Q001,DrugA,0.0,1,100
1,Q002,DrugA,0.0,2,98
2,Q003,DrugA,0.0,3,102
3,Q004,DrugA,0.1,1,87
4,Q005,DrugA,0.1,2,85


In [3]:
# Compute z-score within each drug/concentration group.
df["group_key"] = df["drug_name"] + "_" + df["concentration_uM"].astype(str)

df["group_mean"] = df.groupby("group_key")["cell_viability_percent"].transform("mean")
df["group_sd"] = df.groupby("group_key")["cell_viability_percent"].transform("std")
df["z_score"] = (df["cell_viability_percent"] - df["group_mean"]) / df["group_sd"]

df["qc_flag"] = np.where(df["z_score"].abs() > 1.5, "review", "ok")
df

,sample_id,drug_name,concentration_uM,replicate,cell_viability_percent,group_key,group_mean,group_sd,z_score,qc_flag
0,Q001,DrugA,0.0,1,100,DrugA_0.0,100.000000,2.000000,0.000000,ok
1,Q002,DrugA,0.0,2,98,DrugA_0.0,100.000000,2.000000,-1.000000,ok
2,Q003,DrugA,0.0,3,102,DrugA_0.0,100.000000,2.000000,1.000000,ok
3,Q004,DrugA,0.1,1,87,DrugA_0.1,100.666667,25.423087,-0.537569,ok
4,Q005,DrugA,0.1,2,85,DrugA_0.1,100.666667,25.423087,-0.616238,ok
5,Q006,DrugA,0.1,3,130,DrugA_0.1,100.666667,25.423087,1.153807,ok
6,Q007,DrugA,1.0,1,63,DrugA_1.0,62.666667,2.516611,0.132453,ok
7,Q008,DrugA,1.0,2,60,DrugA_1.0,62.666667,2.516611,-1.059626,ok
8,Q009,DrugA,1.0,3,65,DrugA_1.0,62.666667,2.516611,0.927173,ok
9,Q010,DrugA,10.0,1,30,DrugA_10.0,30.333333,2.516611,-0.132453,ok


In [4]:
fig = px.scatter(
    df,
    x="concentration_uM",
    y="cell_viability_percent",
    color="qc_flag",
    symbol="drug_name",
    hover_data=["sample_id", "drug_name", "replicate", "z_score"],
    title="QC Scatter Plot: Possible Outliers"
)
fig.update_xaxes(type="log")
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


In [5]:
fig2 = px.box(
    df,
    x="drug_name",
    y="cell_viability_percent",
    color="drug_name",
    points="all",
    title="Cell Viability Distribution by Drug"
)
# If this chart does not render in Jupyter, try: fig2.show(renderer="browser")
fig2.show(renderer="iframe")


## Interpretation Questions

1. Which points were flagged for review?
2. Are flagged points always wrong?
3. What should a lab scientist check before removing an outlier?
4. How could this QC view support a future educational BioDose workflow?

## Limitations

- This is synthetic assay data for learning QC ideas.
- A z-score flag is a review prompt, not proof that a point is invalid.
- Real QC decisions require raw data, lab notes, instrument context, and study design details.
